В этом модуле мы поговорим о такой теме, как обработка естественного языка (англ. Natural Language Processing, NLP). NLP изучает проблемы компьютерного анализа естественных языков - т.е. языков, которые для общения используют люди (а не придуманных искусственно (например, азбука Морзе - язык, придуманный искусственно). Подробнее о том, зачем нужен NLP и где именно возникает задача обработки естественного языка мы поговорим в Уроке 1.


# Урок 1. Цели и задачи NLP

Тексты - один из самых доступных и объёмных источников данных. Современный человек потребляет огромное количество контента: фото, звук и текст. Через визуальный канал, т.е. с помощью  фото  и видео, мы получаем 80% всей информации, поэтому такой большой объём курса уделён компьютерному зрению. На втором месте по объёму передаваемой информации - текстовые данные. Кроме того, обработка видео требует мощных вычислительных ресурсов (GPU), а задачи NLP менее прожорливы по памяти -  поэтому давайте поговорим про обработку текста.

Например, если у вас интернет-магазин, то для анализа доступны

* текстовые описания товаров
* пользовательские комментарии
* диалоги с продавцом-консультантом в чатике

Текстовую информацию просто хранить, поэтому проекты накапливают огромные наборы данных такого рода и очень хотят извлекать из этих объёмов полезную информацию.

Как специалист по ML в начале карьеры вы, скорее всего, встретите ряд “классических” задач - например, определение тональности (настроения) текста или классификации сообщений spam/not spam - для таких задач используются подходы, основанные на подсчёте статистик по встречающимся в тексте словам.
Однако, есть и другие, более сложные задачи.

Для решения применяются различные архитектуры нейросетей (RNN, LSTM) - это мощные инструменты, которые позволяют решать сложные задачи, например:

* извлечения именованных сущностей ([NER](https://habr.com/ru/post/414175/), Named-Entity Recognizing)
* автоматизированного перевода (например, сервис *google translate* производит перевод с помощью глубоких сетей)
* Speech Recognition - распознавание речи, трансляция из аудио в текстовый вид
* Natural Language Generation - генерация текстов, например можно генерировать подписи к картинкам

У обработки естественного языка есть ряд особенностей:

* необходимо размечать большой объём данных для обучения с учителем. Допустим, хотим отделять спам-сообщения от не спама. Вам нужно найти людей, которые прочитают все смс, которые удалось собрать и отметят те из них, которые являются спамом - текстов обычно очень много и разметка данных может оказаться дорогим удовольствием.
* модель, обученную на одном языке невозможно использовать для другого языка.
* важен как синтаксис, так и семантика (смысл). Например, во фразе: «Вот списки студентов, которые сдали зачет по физике» определение «которые сдали зачет по физике» относится к студентам, а в предложении: «Вот списки студентов, которые лежали в шкафу у декана»  структура фразы (тот самый синтаксис) такая же, как и предыдущей - но определение уже относится не к студентам, а к листкам бумаги. От компьютера мы хотим добиться, чтобы смыл обеих фраз был “понят” одинаково хорошо.

Кроме того, для текстов на естественном языке довольно сложно проводить предобработку данных, этот этап сильно зависит от задачи, которую вы  решаете. Так, например, для задачи анализа тональности текста знаки препинания, скорее всего, не важны. Однако, для задачи извлечения именованных сущностей (именованная сущность - это имя собственное - например название организации или географического объекта) удалять знаки препинания не рекомендуется - это может привести к потере важной информации. Например если из фразы `Мы пошли обедать в “Берёзку”` если удалить все знаки препинания (кавычки) и заглавную букву в названии заведения то станет сложнее понять, что речь идёт о кафе.

В этом уроке мы узнали о том, какими бывают NLP задачи - в рамках домашней работы мы будем решать задачу поиска похожих слов в текстах твитов. В следующем уроке поговорим о первом этапе решения этой задаче - предварительной обработке текста.

In [1]:
from google.colab import files
import zipfile
import os

#загружаем архив
uploaded = files.upload()

#распаковка
zip_filename = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall('/content/dataset')

#смотрим структуру
!ls -la /content/dataset/


Saving project-4-at-2026-02-20-15-27-04ee68d0.zip to project-4-at-2026-02-20-15-27-04ee68d0.zip
total 40
drwxr-xr-x 4 root root  4096 Feb 20 12:41 .
drwxr-xr-x 1 root root  4096 Feb 20 12:41 ..
-rw-r--r-- 1 root root     6 Feb 20 12:41 classes.txt
drwxr-xr-x 2 root root 12288 Feb 20 12:41 images
drwxr-xr-x 2 root root 12288 Feb 20 12:41 labels
-rw-r--r-- 1 root root   176 Feb 20 12:41 notes.json


In [2]:
#установка Ultralytics и необходимых библиотек
!pip install ultralytics
!pip install roboflow
!pip install albumentations

import ultralytics
ultralytics.checks()

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 21.5/107.7 GB disk)


In [3]:
%%writefile /content/dataset.yaml

#пути к данным
path: /content/dataset  # корневая папка датасета
train: images  # папка с тренировочными изображениями
val: images    # папка с валидационными (пока та же)

#количество классов
nc: 1

#названия классов (должны совпадать с теми, что в разметке!)
names: ['tail']

Writing /content/dataset.yaml


In [5]:
import os
import shutil
import random
from sklearn.model_selection import train_test_split

#создаем структуру папок
os.makedirs('/content/dataset/train/images', exist_ok=True)
os.makedirs('/content/dataset/train/labels', exist_ok=True)
os.makedirs('/content/dataset/val/images', exist_ok=True)
os.makedirs('/content/dataset/val/labels', exist_ok=True)

#получаем список всех изображений
images = [f for f in os.listdir('/content/dataset/images')
          if f.endswith(('.jpg', '.jpeg', '.png'))]

#разделяем на train (80%) и val (20%)
train_images, val_images = train_test_split(images, test_size=0.2, random_state=42)

print(f"Train: {len(train_images)} images")
print(f"Val: {len(val_images)} images")

#копируем файлы в соответствующие папки
for img in train_images:
    #копируем изображение
    shutil.copy(f'/content/dataset/images/{img}', f'/content/dataset/train/images/{img}')
    #копируем соответствующий .txt файл
    txt_file = img.replace('.jpg', '.txt').replace('.jpeg', '.txt').replace('.png', '.txt')
    if os.path.exists(f'/content/dataset/labels/{txt_file}'):
        shutil.copy(f'/content/dataset/labels/{txt_file}', f'/content/dataset/train/labels/{txt_file}')

for img in val_images:
    shutil.copy(f'/content/dataset/images/{img}', f'/content/dataset/val/images/{img}')
    txt_file = img.replace('.jpg', '.txt').replace('.jpeg', '.txt').replace('.png', '.txt')
    if os.path.exists(f'/content/dataset/labels/{txt_file}'):
        shutil.copy(f'/content/dataset/labels/{txt_file}', f'/content/dataset/val/labels/{txt_file}')

Train: 51 images
Val: 13 images


In [6]:
%%writefile /content/dataset.yaml

path: /content/dataset
train: train/images  # теперь указываем подпапки
val: val/images
nc: 1
names: ['tail']

Overwriting /content/dataset.yaml


In [13]:
import cv2
import numpy as np
import albumentations as A
import random
import os
from pathlib import Path
import matplotlib.pyplot as plt

def generate_synthetic_tails(
    source_images_dir,  # папка с исходными фото
    source_labels_dir,  # папка с разметкой
    backgrounds_dir,    # папка с фонами (можно скачать из интернета)
    output_dir,         # куда сохранять
    num_synthetic=100   # сколько генерировать
):
    """
    Генерирует синтетические данные: хвосты на случайных фонах
    """
    os.makedirs(f"{output_dir}/images", exist_ok=True)
    os.makedirs(f"{output_dir}/labels", exist_ok=True)

    #загружаем все исходные хвосты
    source_images = list(Path(source_images_dir).glob('*.jpg')) + list(Path(source_images_dir).glob('*.png'))
    print(f"Найдено исходных изображений: {len(source_images)}")

    #загружаем фоны
    backgrounds = list(Path(backgrounds_dir).glob('*.jpg')) + list(Path(backgrounds_dir).glob('*.png'))
    print(f"Найдено фонов: {len(backgrounds)}")

    if not backgrounds:
        print("Нет фонов")
        return

    #аугментации для хвоста перед вставкой
    tail_transform = A.Compose([
        A.Rotate(limit=45, p=0.8),           # поворот
        A.RandomScale(scale_limit=0.3, p=0.7), # масштаб
        A.RandomBrightnessContrast(p=0.5),    # яркость/контраст
        A.HueSaturationValue(p=0.3),          # цвет
        A.Blur(blur_limit=3, p=0.2),           # размытие
    ])

    for i in range(num_synthetic):
        #выбираем случайный исходный хвост
        src_path = random.choice(source_images)
        src_img = cv2.imread(str(src_path))
        src_img = cv2.cvtColor(src_img, cv2.COLOR_BGR2RGB)
        h_src, w_src = src_img.shape[:2]

        #загружаем разметку для этого хвоста
        label_path = Path(source_labels_dir) / (src_path.stem + '.txt')
        if not label_path.exists():
            continue

        with open(label_path, 'r') as f:
            line = f.readline().strip()
            parts = line.split()
            class_id = parts[0]
            coords = list(map(float, parts[1:]))

        #создаем маску хвоста
        mask = np.zeros((h_src, w_src), dtype=np.uint8)
        points = []
        for j in range(0, len(coords), 2):
            x = int(coords[j] * w_src)
            y = int(coords[j+1] * h_src)
            points.append([x, y])
        points = np.array(points, dtype=np.int32)
        cv2.fillPoly(mask, [points], 255)

        #извлекаем только хвост с его маской
        tail_only = cv2.bitwise_and(src_img, src_img, mask=mask)

        #выбираем случайный фон
        bg_path = random.choice(backgrounds)
        bg = cv2.imread(str(bg_path))
        bg = cv2.cvtColor(bg, cv2.COLOR_BGR2RGB)
        bg = cv2.resize(bg, (640, 640))  # фиксируем размер

        #применяем аугментации к хвосту
        augmented = tail_transform(image=tail_only, mask=mask)
        tail_aug = augmented['image']
        mask_aug = augmented['mask']

        #находим контур хвоста после аугментаций
        contours, _ = cv2.findContours(mask_aug, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue

        #берем самый большой контур
        tail_contour = max(contours, key=cv2.contourArea)

        #определяем позицию вставки (случайно)
        h_tail, w_tail = tail_aug.shape[:2]
        max_x = bg.shape[1] - w_tail
        max_y = bg.shape[0] - h_tail

        if max_x < 10 or max_y < 10:
            continue

        pos_x = random.randint(10, max_x)
        pos_y = random.randint(10, max_y)

        #вставляем хвост на фон
        result = bg.copy()
        roi = result[pos_y:pos_y+h_tail, pos_x:pos_x+w_tail]

        #создаем маску для наложения
        mask_3channel = cv2.cvtColor(mask_aug, cv2.COLOR_GRAY2BGR) / 255.0

        #наложение
        blended = (roi * (1 - mask_3channel) + tail_aug * mask_3channel).astype(np.uint8)
        result[pos_y:pos_y+h_tail, pos_x:pos_x+w_tail] = blended

        #создаем новую разметку для синтетического изображения
        new_coords = []
        for point in tail_contour:
            x_point, y_point = point[0]
            #глобальные координаты на фоне
            global_x = (pos_x + x_point) / bg.shape[1]
            global_y = (pos_y + y_point) / bg.shape[0]
            new_coords.extend([global_x, global_y])

        #сохраняем
        out_img_path = f"{output_dir}/images/synth_{i:04d}.jpg"
        out_label_path = f"{output_dir}/labels/synth_{i:04d}.txt"

        cv2.imwrite(out_img_path, cv2.cvtColor(result, cv2.COLOR_RGB2BGR))

        with open(out_label_path, 'w') as f:
            line = f"{class_id} " + " ".join([f"{c:.6f}" for c in new_coords])
            f.write(line)

        if i % 10 == 0:
            print(f"Сгенерировано {i}/{num_synthetic}")

    print(f" Сгенерировано {num_synthetic} синтетических изображений")

#использование:
generate_synthetic_tails(
    source_images_dir="/content/dataset/images",
    source_labels_dir="/content/dataset/labels",
    backgrounds_dir="/path/to/backgrounds",  # нужно создать папку с фонами
    output_dir="/content/dataset_synthetic",
    num_synthetic=100
)

Найдено исходных изображений: 64
Найдено фонов: 0
Нет фонов


In [16]:
#скачиваем COCO-128, которые можно использовать как фоны
!wget -O /content/coco128.zip https://github.com/ultralytics/yolov5/releases/download/v1.0/coco128.zip
!unzip /content/coco128.zip -d /content/backgrounds_coco/

--2026-02-20 13:03:18--  https://github.com/ultralytics/yolov5/releases/download/v1.0/coco128.zip
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/264818686/854f8531-cc3e-47d1-9f20-5d8fa189e18a?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-02-20T13%3A37%3A00Z&rscd=attachment%3B+filename%3Dcoco128.zip&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-02-20T12%3A36%3A13Z&ske=2026-02-20T13%3A37%3A00Z&sks=b&skv=2018-11-09&sig=DG%2FXCsKExArI5jnqdEW%2Bby3Nqfee5OTaumWbRz71QDs%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MTU5Mjg5OSwibmJmIjoxNzcxNTkyNTk5LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5ibG9iLmNvc

In [18]:
import os
import zipfile
from pathlib import Path

#путь к скачанному архиву
zip_path = "/content/coco128.zip"
# Папка для распаковки
extract_path = "/content/backgrounds_coco"

#создаем папку, если её нет
os.makedirs(extract_path, exist_ok=True)

#распаковываем архив
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"Архив распакован в {extract_path}")

Архив распакован в /content/backgrounds_coco


In [19]:
backgrounds_dir = "/content/backgrounds_coco/coco128/images/train2017/"

#проверяем, существует ли папка
if os.path.exists(backgrounds_dir):
    bg_files = os.listdir(backgrounds_dir)
    print(f"Найдено фонов в {backgrounds_dir}: {len(bg_files)}")
    print(f"Примеры: {bg_files[:5]}")
else:
    #если структура другая, ищем все .jpg файлы рекурсивно
    bg_files = list(Path("/content/backgrounds_coco").rglob("*.jpg"))
    print(f"Найдено .jpg файлов всего: {len(bg_files)}")
    if bg_files:
        backgrounds_dir = str(bg_files[0].parent)
        print(f"Фоны находятся в: {backgrounds_dir}")

Найдено фонов в /content/backgrounds_coco/coco128/images/train2017/: 128
Примеры: ['000000000490.jpg', '000000000349.jpg', '000000000370.jpg', '000000000307.jpg', '000000000436.jpg']


In [20]:
#создаем простую структуру: одна папка со всеми фонами
!mkdir -p /content/backgrounds

#копируем все изображения в одну папку (если нашли)
if 'bg_files' in locals() and bg_files:
    for i, bg_file in enumerate(bg_files):
        !cp "{bg_file}" /content/backgrounds/
    print(f"Скопировано {len(bg_files)} фонов в /content/backgrounds/")
else:
    print("Не удалось найти файлы. Проверим структуру вручную:")
    !find /content/backgrounds_coco -type f -name "*.jpg" | head -10

cp: cannot stat '000000000490.jpg': No such file or directory
cp: cannot stat '000000000349.jpg': No such file or directory
cp: cannot stat '000000000370.jpg': No such file or directory
cp: cannot stat '000000000307.jpg': No such file or directory
cp: cannot stat '000000000436.jpg': No such file or directory
cp: cannot stat '000000000529.jpg': No such file or directory
cp: cannot stat '000000000472.jpg': No such file or directory
cp: cannot stat '000000000260.jpg': No such file or directory
cp: cannot stat '000000000389.jpg': No such file or directory
cp: cannot stat '000000000241.jpg': No such file or directory
cp: cannot stat '000000000581.jpg': No such file or directory
cp: cannot stat '000000000034.jpg': No such file or directory
cp: cannot stat '000000000312.jpg': No such file or directory
cp: cannot stat '000000000042.jpg': No such file or directory
cp: cannot stat '000000000201.jpg': No such file or directory
cp: cannot stat '000000000443.jpg': No such file or directory
cp: cann

In [23]:
generate_synthetic_tails(
    source_images_dir="/content/dataset/images",
    source_labels_dir="/content/dataset/labels",
    backgrounds_dir="/content/backgrounds_coco/coco128/images/train2017",  # наша папка со всеми фонами
    output_dir="/content/dataset_synthetic",
    num_synthetic=100  # сколько генерировать
)

Найдено исходных изображений: 64
Найдено фонов: 128
Сгенерировано 0/100
Сгенерировано 10/100
Сгенерировано 20/100
Сгенерировано 30/100
Сгенерировано 40/100
Сгенерировано 50/100
Сгенерировано 60/100
Сгенерировано 70/100
Сгенерировано 80/100
Сгенерировано 90/100
 Сгенерировано 100 синтетических изображений


In [24]:
import os
import shutil

#создаем папку для финального датасета
!mkdir -p /content/final_dataset/images
!mkdir -p /content/final_dataset/labels

#копируем реальные данные
!cp /content/dataset/train/images/* /content/final_dataset/images/ 2>/dev/null
!cp /content/dataset/train/labels/* /content/final_dataset/labels/ 2>/dev/null

#копируем валидационные данные
!cp /content/dataset/val/images/* /content/final_dataset/images/ 2>/dev/null
!cp /content/dataset/val/labels/* /content/final_dataset/labels/ 2>/dev/null

#копируем синтетические данные
!cp /content/dataset_synthetic/images/* /content/final_dataset/images/
!cp /content/dataset_synthetic/labels/* /content/final_dataset/labels/

print(f"Всего изображений в финальном датасете: {len(os.listdir('/content/final_dataset/images'))}")
print(f"Всего файлов разметки: {len(os.listdir('/content/final_dataset/labels'))}")

Всего изображений в финальном датасете: 164
Всего файлов разметки: 164


In [25]:
%%writefile /content/dataset_updated.yaml

#пути к данным
path: /content/final_dataset  # корневая папка с объединенными данными
train: images  # все изображения в одной папке
val: images    # пока валидация там же (

# Количество классов
nc: 1

# Названия классов
names: ['tail']

Writing /content/dataset_updated.yaml


In [26]:
from sklearn.model_selection import train_test_split
import os
import shutil

#создаем структуру
!mkdir -p /content/final_split/train/images
!mkdir -p /content/final_split/train/labels
!mkdir -p /content/final_split/val/images
!mkdir -p /content/final_split/val/labels

#получаем все изображения
all_images = os.listdir('/content/final_dataset/images')

#отделяем реальные от синтетических
real_images = [f for f in all_images if not f.startswith('synth_')]
synthetic_images = [f for f in all_images if f.startswith('synth_')]

print(f"Реальных: {len(real_images)}")
print(f"Синтетических: {len(synthetic_images)}")

#реальные делим 80/20
real_train, real_val = train_test_split(real_images, test_size=0.2, random_state=42)

#синтетические все идут в train
synth_train = synthetic_images

#объединяем
train_images = real_train + synth_train
val_images = real_val

print(f"Train: {len(train_images)} изображений")
print(f"Val: {len(val_images)} изображений")

#копируем train
for img in train_images:
    # Копируем изображение
    shutil.copy(f'/content/final_dataset/images/{img}', f'/content/final_split/train/images/{img}')
    # Копируем соответствующий .txt
    txt_file = img.replace('.jpg', '.txt').replace('.jpeg', '.txt').replace('.png', '.txt')
    if os.path.exists(f'/content/final_dataset/labels/{txt_file}'):
        shutil.copy(f'/content/final_dataset/labels/{txt_file}', f'/content/final_split/train/labels/{txt_file}')

#копируем val
for img in val_images:
    shutil.copy(f'/content/final_dataset/images/{img}', f'/content/final_split/val/images/{img}')
    txt_file = img.replace('.jpg', '.txt').replace('.jpeg', '.txt').replace('.png', '.txt')
    if os.path.exists(f'/content/final_dataset/labels/{txt_file}'):
        shutil.copy(f'/content/final_dataset/labels/{txt_file}', f'/content/final_split/val/labels/{txt_file}')

print("Данные разделены!")

Реальных: 64
Синтетических: 100
Train: 151 изображений
Val: 13 изображений
Данные разделены!


In [27]:
%%writefile /content/dataset_final.yaml

path: /content/final_split
train: train/images
val: val/images
nc: 1
names: ['tail']

Writing /content/dataset_final.yaml


In [30]:
from google.colab import drive
import os

#монтируем Google Drive
drive.mount('/content/drive')

#создаем папку для проектов (если её нет)
!mkdir -p /content/drive/MyDrive/yolo_projects/tail_segmentation




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:

#импортируем и обучаем
from ultralytics import YOLO

#загружаем предобученную модель
model = YOLO('yolo11n-seg.pt')

#обучаем
results = model.train(
    data='/content/dataset_final.yaml',  # путь к датасету
    epochs=100,
    imgsz=640,
    batch=8,
    workers=4,
    patience=20,
    save=True,
    device='cuda',

    #аугментации для хвостов
    augment=True,
    degrees=30.0,        # поворот
    translate=0.2,       # сдвиг
    brightness=0.3,      # яркость
    contrast=0.3,        # контраст
    scale=0.5,           # масштаб
    shear=10.0,          # сдвиг
    mosaic=1.0,          # склейка
    mixup=0.3,           # смешивание
    copy_paste=0.4,      # копирование
    fliplr=0.5,          # горизонтальный флип
    flipud=0.1,          # вертикальный флип
    perspective=0.0002,  # перспектива
    saturation=0.3,      # насыщенность
    hue=0.1,             # оттенок
)

#копируем только лучшую модель в Google Drive
import shutil
shutil.copy(
    '/content/runs/segment/train/weights/best.pt',
    '/content/drive/MyDrive/yolo_projects/tail_segmentation/tail_model.pt'
)

print("Модель сохранена в Google Drive:")
print("   /content/drive/MyDrive/yolo_projects/tail_segmentation/tail_model.pt")
print(f"mAP50-95: {results.box.map:.4f}")

ModuleNotFoundError: No module named 'ultralytics'

In [ ]:
# Визуализация метрик
from IPython.display import Image
Image(filename='/content/runs/segment/train/results.png', width=800)

# Метрики на валидационной выборке
metrics = model.val()
print(f"mAP50-95: {metrics.box.map:.4f}")

In [ ]:
# Загружаем лучшею сохраненную модель
best_model = YOLO('/content/runs/segment/train/weights/best.pt')

# Тестируем на валидационных изображениях
for img in val_images[:3]:  # первые 3 из валидации
    results = best_model(f'/content/dataset/val/images/{img}')

    plt.figure(figsize=(12, 6))
    plt.imshow(results[0].plot())
    plt.title(f"Prediction on: {img}")
    plt.axis('off')
    plt.show()

In [ ]:
# Копируем лучшую модель в корневую папку
!cp /content/runs/segment/train/weights/best.pt /content/tail_segmentation_model.pt

# Скачиваем на компьютер
from google.colab import files
files.download('/content/tail_segmentation_model.pt')

In [8]:
import os

# Посмотрим содержимое одного .txt файла
labels_folder = '/content/dataset/labels/'
txt_files = os.listdir(labels_folder)

if txt_files:
    # Берем первый файл
    sample_txt = txt_files[0]
    print(f"Смотрим файл: {sample_txt}")
    print("-" * 50)

    with open(os.path.join(labels_folder, sample_txt), 'r') as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        parts = line.strip().split()
        class_id = parts[0]
        coords = parts[1:]

        print(f"Объект {i+1}:")
        print(f"  Class ID: {class_id}")
        print(f"  Координат: {len(coords)}")
        print(f"  Первые 10 координат: {coords[:10]}")
        print()

        # Проверяем, что за класс
        if class_id == '0':
            print("  ✅ Это должен быть 'tail' (ID 0)")
        else:
            print(f"  ❌ Неизвестный класс ID: {class_id}")
else:
    print("Нет .txt файлов в папке!")

Смотрим файл: 77504b36-a658d265d250b41b9b6681e1d8315e60_m.txt
--------------------------------------------------
Объект 1:
  Class ID: 0
  Координат: 10
  Первые 10 координат: ['0.6512968299711815', '0.8508703170028817', '0.6930835734870316', '0.8328760806916427', '0.7680115273775217', '0.8277348703170029', '0.760806916426513', '0.8585821325648415', '0.7247838616714697', '0.8637233429394813']

  ✅ Это должен быть 'tail' (ID 0)
